# 02 · Fine-tune Qwen2.5-Coder (QLoRA) for Text2SQL — Colab T4

Free-tier recipe: **Unsloth + QLoRA** on `Qwen2.5-Coder-1.5B-Instruct`
(Apache-2.0, 1.54B params, code-specialised). ~1.5–2 GB VRAM for the adapter;
fits a free T4 with room to spare.

> **First:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(enable the T4 GPU runtime!)')

## 1. Get the code (clone the repo)

In [ ]:
# --- SETUP: run me first (works on Colab AND locally) ---
import os, sys, subprocess
if not os.path.exists('src') and os.path.basename(os.getcwd()) != 'text2sql-finetuning':
    if not os.path.isdir('text2sql-finetuning'):
        subprocess.run(['git', 'clone', 'https://github.com/Shiverion/text2sql-finetuning.git'], check=True)
    os.chdir('text2sql-finetuning')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())
print('src present :', os.path.isdir('src'))

## 2. Install Unsloth + training stack
`pip install unsloth` pulls a torch/transformers/trl/peft combo that's tested
together — don't hand-pin these on Colab unless you know why. (~2–4 min.)

In [ ]:
!pip install -q unsloth
# If you hit a version error later, get the matching nightly instead:
# !pip install -q --upgrade --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git" \
#     "unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git"

## 3. Download BIRD
If you'd rather skip this, upload your own processed JSONL and jump to step 5.

In [ ]:
!wget -q https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip && unzip -q -o train.zip
!wget -q https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip   && unzip -q -o dev.zip

### Check the extracted layout
BIRD's archive layout shifts between releases. Look at the tree below and set
`TRAIN_JSON / TRAIN_DB / DEV_JSON / DEV_DB` to match what you actually see
(you want the folder that contains one sub-folder per database, each holding
`<db_id>.sqlite`).

In [ ]:
import glob, os
for p in sorted(glob.glob('train/*'))[:10] + sorted(glob.glob('dev*/*'))[:10]:
    print(p)
print('--- sample sqlite files ---')
for p in glob.glob('**/*.sqlite', recursive=True)[:5]:
    print(p)

In [ ]:
# Edit these to match the tree printed above:
TRAIN_JSON = 'train/train.json'
TRAIN_DB   = 'train/train_databases'
DEV_JSON   = 'dev/dev.json'
DEV_DB     = 'dev/dev_databases'

## 4. Preprocess → JSONL
Builds prompt/messages records with the schema reconstructed from each .sqlite.

In [ ]:
!python -m src.data_prep --source bird --json {TRAIN_JSON} --db_root {TRAIN_DB} \
    --out data/processed/bird_train.jsonl --shuffle
!python -m src.data_prep --source bird --json {DEV_JSON} --db_root {DEV_DB} \
    --out data/processed/bird_dev.jsonl

In [ ]:
# peek at one processed record
import json
print(json.dumps(json.loads(open('data/processed/bird_train.jsonl').readline()), indent=2)[:1200])

## 5. Train (QLoRA)
Runs the hardened trainer in `src/train.py` (Unsloth + completion-only loss;
the SFTConfig/SFTTrainer kwargs auto-adapt to the installed TRL version).
`--max_train_samples` keeps a free-tier run inside the session limit; drop it
for the full set. For a 2-minute sanity check first, add `--max_steps 30`.

In [ ]:
!python -m src.train --preset exp1_qwen1.5b_bird_qlora \
    --train_file data/processed/bird_train.jsonl \
    --val_file   data/processed/bird_dev.jsonl \
    --max_train_samples 8000 --epochs 2

The adapter + tokenizer are saved to `outputs/qwen2.5-coder-1.5b-bird-qlora/`.
The training implementation lives in [`src/train.py`](src/train.py); tweak
hyper-parameters via the presets in [`src/config.py`](src/config.py).

## 6. (Optional) push the adapter to the Hub

In [ ]:
# from huggingface_hub import login; login()
# from unsloth import FastLanguageModel  # already imported during training
# model.push_to_hub_merged(...)  # or just upload the outputs/ folder
# !huggingface-cli upload <you>/qwen2.5-coder-1.5b-bird-qlora outputs/qwen2.5-coder-1.5b-bird-qlora

Now run **03_inference_eval.ipynb** to measure execution accuracy.